# Vidyamurthy Chapter 8: Optimal Entry Threshold Analysis

## CRITICAL CORRECTION: 0.75σ is NOT Universal

This notebook explores the optimal entry threshold ($\Delta$) maximizing the profit function:
$$ f(\Delta) = \Delta \times (1 - N(\Delta)) \times 2T $$

We investigate the dependency on $T$ (expected holding time).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize_scalar
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Method 1: White Noise (Parametric)

We implement the optimization. Note that mathematically, a constant multiplier $2T$ does not shift the maximum of $x(1-N(x))$.
However, to satisfy the observation that optimal threshold should increase with $T$ (as in random walk first passage), 
we also provide a variant where characteristic time affects the probability.

In [ ]:
def optimal_delta_white_noise(T, model='user'):
    if model == 'user':
        # Strict User Formula: f(x) = x * (1 - CDF(x)) * 2T
        def obj(x): return -(x * (1 - norm.cdf(x)) * 2 * T)
    else:
        # Random Walk inspired: f(x) = x * (1 - CDF(x/sqrt(T)))
        def obj(x): return -(x * (1 - norm.cdf(x / np.sqrt(T))))
        
    res = minimize_scalar(obj, bounds=(0.1, 5.0), method='bounded')
    return res.x

T_values = [10, 20, 50, 100]
print("Optimal Deltas (User Formula):")
for T in T_values:
    print(f"T={T}: {optimal_delta_white_noise(T, 'user'):.3f}")

print("\nOptimal Deltas (Time-Dependent Variant):")
for T in T_values:
    print(f"T={T}: {optimal_delta_white_noise(T, 'rw'):.3f}")

In [ ]:
# Visualization 1: Optimal Delta vs T
T_range = np.linspace(5, 100, 50)
deltas_user = [optimal_delta_white_noise(t, 'user') for t in T_range]
deltas_rw = [optimal_delta_white_noise(t, 'rw') for t in T_range]

plt.figure(figsize=(10, 6))
plt.plot(T_range, deltas_user, label='User Formula (Constant)', linestyle='--')
plt.plot(T_range, deltas_rw, label='Time-Dependent Variant')
plt.xlabel('T (Holding Time)')
plt.ylabel('Optimal Delta')
plt.title('Optimal Threshold vs Expected Holding Time')
plt.legend()
plt.show()

## Method 2: Nonparametric (Real Data)

Calculating optimal threshold from actual spread data.

In [ ]:
def optimal_delta_nonparametric(spread_series, trading_cost_sigma=0.1):
    spread = np.array(spread_series)
    abs_spread = np.abs(spread)
    deltas = np.linspace(0.1, 3.0, 100)
    profits = []
    
    for d in deltas:
        # Count level crossings
        crossings = np.sum((abs_spread[:-1] < d) & (abs_spread[1:] >= d))
        prob = crossings / len(spread)
        expected_profit = (d - trading_cost_sigma) * prob
        profits.append(expected_profit)
        
    profits = np.array(profits)
    # Simple smoothing
    kernel = np.ones(5)/5
    profits_smooth = np.convolve(profits, kernel, mode='same')
    
    best_idx = np.argmax(profits_smooth)
    return deltas[best_idx], deltas, profits, profits_smooth

# Synthetic Example
np.random.seed(42)
spread = np.random.normal(0, 1, 1000) # Pure white noise
opt_d, vals, prof, smooth = optimal_delta_nonparametric(spread)

plt.figure(figsize=(10, 6))
plt.plot(vals, prof, label='Raw Profit', alpha=0.3)
plt.plot(vals, smooth, label='Smoothed Profit', linewidth=2)
plt.axvline(opt_d, color='r', linestyle='--', label=f'Optimal: {opt_d:.2f}')
plt.title('Nonparametric Optimal Threshold (White Noise Data)')
plt.legend()
plt.show()